In [2]:
import arcpy
from pathlib import Path
import os
import shutil
import pandas as pd
from datetime import datetime
import re
import unicodedata


In [1]:
# =============================================================================
# PATHS BASE - CL_MLP_PAO
# =============================================================================

# Carpeta de entrada donde se dejan los vuelos de drone sin procesar
PATH_INPUT_VUELOS_DRONE = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"


# =============================================================================
# PROYECTO ARCGIS PRO
# =============================================================================

# Proyecto APRX del Visor Territorial SIG PAO
PATH_APRX_VISOR_TERRITORIAL = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"


# =============================================================================
# GEODATABASES
# =============================================================================

# Geodatabase de imágenes PAO
PATH_GDB_IMAGENES = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"

# Geodatabase principal PAO v1
PATH_GDB_PAO_V1 = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"


# =============================================================================
# FEATURE CLASSES - IMÁGENES
# =============================================================================

# Feature class de imágenes oblicuas de drone
PATH_FC_IMAGENES_OBLICUAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"
    r"\CL_MLP_PAO_01_Imagenes_Oblicuas\CL_MLP_PAO_Oblicuas_Drone_v2"
)

# Índice / diccionario de vuelos PAO para imágenes
PATH_FC_INDICE_VUELOS_IMGS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
)


# =============================================================================
# FEATURE CLASSES - COMPLEMENTOS
# =============================================================================

# Feature class de macrozonas PAO
PATH_FC_MACROZONAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_ZONAS_Macrozonas_PO"
)


# =============================================================================
# FEATURE CLASSES - VIDEOS
# =============================================================================

# Feature class de videos de drone en terreno
PATH_FC_VIDEOS_DRONE = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_07_IMAGENES_TERRENO\MLP_SIG_PAO_Videos_Drone"
)


# =============================================================================
# CARPETAS DE VUELOS PROCESADOS / FECHA 26_06
# =============================================================================

# Carpeta de vuelos Drone - Chacay
PATH_DRONE_CHACAY_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_Drone\26_06"

# Carpeta de vuelos Drone - Chacay El Mauro
PATH_DRONE_CHACAY_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro
PATH_DRONE_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro Puerto Punta Chungo
PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Puerto_Punta_Chungo_Drone\26_06"

# Carpeta de vuelos Drone - Puerto Punta Chungo
PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Puerto_Punta_Chungo_Drone\26_06"


# =============================================================================
# LISTAS AGRUPADAS
# =============================================================================

# Lista de carpetas de vuelos por zona
PATHS_CARPETAS_VUELOS_DRONE = [
    PATH_DRONE_CHACAY_2606,
    PATH_DRONE_CHACAY_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606,
    PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606,
]

# Diccionario general de paths principales
PATHS_PAO = {
    "input_vuelos_drone": PATH_INPUT_VUELOS_DRONE,
    "aprx_visor_territorial": PATH_APRX_VISOR_TERRITORIAL,
    "gdb_imagenes": PATH_GDB_IMAGENES,
    "gdb_pao_v1": PATH_GDB_PAO_V1,
    "fc_imagenes_oblicuas": PATH_FC_IMAGENES_OBLICUAS,
    "fc_indice_vuelos_imgs": PATH_FC_INDICE_VUELOS_IMGS,
    "fc_macrozonas": PATH_FC_MACROZONAS,
    "fc_videos_drone": PATH_FC_VIDEOS_DRONE,
    "carpetas_vuelos_drone": PATHS_CARPETAS_VUELOS_DRONE,
}

In [ ]:
# =============================================================================
# FASE 1 - CONFIGURACION DE CARGA DE IMAGENES
# =============================================================================

# Carpeta oficial donde llegan los vuelos drone que deben evaluarse para cargar al mosaico.
PATH_INPUT_SCAN = Path(PATH_INPUT_VUELOS_DRONE)

IMAGE_EXTENSIONS = {
    ".tif",
    ".tiff",
    ".jpg",
    ".jpeg",
    ".png",
    ".sid",
    ".jp2",
    ".ecw",
}

PATH_INPUT_SCAN

## Fase 1.1 - Buscar imagenes nuevas en carpeta de vuelos drone


In [ ]:
def scan_input_images(input_folder, extensions=None):
    input_folder = Path(input_folder)
    extensions = {ext.lower() for ext in (extensions or IMAGE_EXTENSIONS)}

    if not input_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta input: {input_folder}")

    rows = []

    for file_path in sorted(input_folder.rglob("*")):
        if not file_path.is_file() or file_path.suffix.lower() not in extensions:
            continue

        stat = file_path.stat()
        rows.append(
            {
                "file_name": file_path.name,
                "stem": file_path.stem,
                "extension": file_path.suffix.lower(),
                "path": str(file_path),
                "relative_path": str(file_path.relative_to(input_folder)),
                "size_mb": round(stat.st_size / (1024 * 1024), 3),
                "modified_at": datetime.fromtimestamp(stat.st_mtime),
            }
        )

    return pd.DataFrame(rows)


input_images_df = scan_input_images(PATH_INPUT_SCAN)
print(f"Carpeta evaluada: {PATH_INPUT_SCAN}")
print(f"Imagenes encontradas en entrada de vuelos drone: {len(input_images_df)}")
input_images_df.head(20)

## Fase 1.2 - Revisar campos del mosaic dataset


In [ ]:
def list_dataset_fields(dataset_path):
    fields = arcpy.ListFields(dataset_path)
    return pd.DataFrame(
        [
            {
                "name": field.name,
                "alias": field.aliasName,
                "type": field.type,
                "length": field.length,
                "required": field.required,
                "editable": field.editable,
            }
            for field in fields
        ]
    )


mosaic_fields_df = list_dataset_fields(PATH_MOSAIC_DATASET)
print(f"Campos del mosaic dataset: {len(mosaic_fields_df)}")
mosaic_fields_df

## Fase 1.3 - Crear DataFrame del mosaic dataset


In [ ]:
def detect_candidate_path_fields(fields_df):
    tokens = ("path", "uri", "url", "file", "name", "source", "raster")
    candidate_fields = []

    for _, row in fields_df.iterrows():
        field_name = row["name"]
        field_type = row["type"]
        normalized_name = field_name.lower()

        if field_type in ("String", "Guid") and any(token in normalized_name for token in tokens):
            candidate_fields.append(field_name)

    return candidate_fields


def table_to_dataframe(dataset_path, fields=None, max_rows=5000):
    if fields is None:
        fields = [field.name for field in arcpy.ListFields(dataset_path) if field.type not in ("Geometry", "Raster", "Blob")]

    rows = []

    with arcpy.da.SearchCursor(dataset_path, fields) as cursor:
        for index, values in enumerate(cursor):
            if index >= max_rows:
                break

            rows.append(dict(zip(fields, values)))

    return pd.DataFrame(rows)


candidate_path_fields = detect_candidate_path_fields(mosaic_fields_df)
print("Campos candidatos para path/name del raster:", candidate_path_fields)

mosaic_df = table_to_dataframe(PATH_MOSAIC_DATASET, max_rows=5000)
print(f"Registros leidos del mosaic dataset: {len(mosaic_df)}")
mosaic_df.head(20)

## Fase 1.4 - Comparar vuelos drone de entrada contra el mosaic dataset


In [ ]:
def normalize_image_key(value):
    if value is None:
        return None

    value = str(value).strip().replace("/", "\\")

    if not value:
        return None

    return value.lower()


def extract_mosaic_path_parts(value):
    normalized_value = normalize_image_key(value)

    if not normalized_value:
        return {
            "mosaic_path": None,
            "mosaic_file_name": None,
            "mosaic_stem": None,
            "mosaic_key": None,
            "mosaic_file_key": None,
            "mosaic_stem_key": None,
        }

    mosaic_path = normalized_value

    if "file?id=" in mosaic_path:
        mosaic_path = mosaic_path.split("file?id=", 1)[1]

    if "&" in mosaic_path:
        mosaic_path = mosaic_path.split("&", 1)[0]

    mosaic_path = mosaic_path.strip()
    mosaic_file_name = re.split(r"[\\/]", mosaic_path)[-1] if mosaic_path else None
    mosaic_stem = Path(mosaic_file_name).stem if mosaic_file_name else None

    return {
        "mosaic_path": mosaic_path,
        "mosaic_file_name": mosaic_file_name,
        "mosaic_stem": mosaic_stem,
        "mosaic_key": mosaic_path,
        "mosaic_file_key": mosaic_file_name.lower() if mosaic_file_name else None,
        "mosaic_stem_key": mosaic_stem.lower() if mosaic_stem else None,
    }


def build_mosaic_image_inventory(mosaic_df, candidate_fields):
    inventory_rows = []

    for field in candidate_fields:
        if field not in mosaic_df.columns:
            continue

        for row_index, value in mosaic_df[field].dropna().items():
            path_parts = extract_mosaic_path_parts(value)

            if not path_parts["mosaic_key"]:
                continue

            inventory_rows.append(
                {
                    "mosaic_row_index": row_index,
                    "source_field": field,
                    "source_value": value,
                    **path_parts,
                }
            )

    inventory_df = pd.DataFrame(inventory_rows)

    if inventory_df.empty:
        return inventory_df

    return inventory_df.drop_duplicates(subset=["source_field", "mosaic_key", "mosaic_file_key", "mosaic_stem_key"])


def build_mosaic_lookup_from_inventory(mosaic_inventory_df):
    lookup = set()

    if mosaic_inventory_df.empty:
        return lookup

    for column in ["mosaic_key", "mosaic_file_key", "mosaic_stem_key"]:
        for value in mosaic_inventory_df[column].dropna():
            normalized_value = normalize_image_key(value)

            if normalized_value:
                lookup.add(normalized_value)

    return lookup


mosaic_image_inventory_df = build_mosaic_image_inventory(mosaic_df, candidate_path_fields)
mosaic_lookup = build_mosaic_lookup_from_inventory(mosaic_image_inventory_df)

print(f"Registros/path candidatos extraidos del mosaic dataset: {len(mosaic_image_inventory_df)}")
display(mosaic_image_inventory_df.head(50))

# Comparacion inicial usando nombres/path existentes del mosaic dataset.

if input_images_df.empty:
    input_images_df = input_images_df.assign(exists_in_mosaic=[], match_key=[])
else:
    input_images_df = input_images_df.copy()
    input_images_df["match_key"] = input_images_df["path"].map(normalize_image_key)
    input_images_df["exists_in_mosaic"] = input_images_df.apply(
        lambda row: any(
            key in mosaic_lookup
            for key in (
                normalize_image_key(row["path"]),
                normalize_image_key(row["file_name"]),
                normalize_image_key(row["stem"]),
            )
            if key
        ),
        axis=1,
    )

new_images_df = input_images_df[input_images_df["exists_in_mosaic"] == False].copy()

print(f"Imagenes en input: {len(input_images_df)}")
print(f"Imagenes ya detectadas en mosaic dataset: {int(input_images_df['exists_in_mosaic'].sum()) if not input_images_df.empty else 0}")
print(f"Imagenes nuevas candidatas a cargar: {len(new_images_df)}")

new_images_df

## Fase 1.5 - Aplicar logica de renombre en Python

In [ ]:
# Parametros equivalentes a la logica historica del Excel, ahora en Python.
RENAME_PREFIX = "CL_MLP_PAO_IF_Ortho"
DEFAULT_RENAMED_EXTENSION = ".tif"

# Casos especiales conocidos. Agregar aqui excepciones cuando el nombre original no trae
# suficiente informacion para inferir el sector exacto usado en el mosaic dataset.
SECTOR_ALIASES = {
    "ESTACION DE VALVULAS N 2": "EV2",
    "ESTACION DE VALVULAS N2": "EV2",
    "ESTACION DE VALVULAS 2": "EV2",
    "ESTACION DISIPADORA TERMINAL": "EDT",
    "ACCESO A POZOS": "Acceso_a_pozos",
    "POZOS PRP": "Pozos_PRP",
    "CAMINO ALTERNATIVO SALAMANCA": "Camino_alternativo_Salamanca",
}

print(RENAME_PREFIX, DEFAULT_RENAMED_EXTENSION)

In [ ]:
def strip_accents(value):
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(char for char in value if not unicodedata.combining(char))


def normalize_sector_text(value):
    value = strip_accents(value)
    value = value.replace("°", " ").replace("º", " ")
    value = re.sub(r"[^0-9A-Za-z]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def format_sector_token(value):
    value = normalize_sector_text(value)

    if not value:
        return None

    alias_key = value.upper()
    if alias_key in SECTOR_ALIASES:
        return SECTOR_ALIASES[alias_key]

    return value.replace(" ", "_")


def extract_date_token_from_filename(file_name):
    stem = Path(file_name).stem

    patterns = [
        r"(?P<year>20\d{2})(?P<month>\d{2})(?P<day>\d{2})",
        r"(?P<day>\d{2})[-_](?P<month>\d{2})[-_](?P<year>20\d{2})",
        r"(?P<day>\d{2})[-_](?P<month>\d{2})[-_](?P<year>\d{2})",
        r"(?P<day>\d{2})(?P<month>\d{2})(?P<year>\d{2})(?!\d)",
    ]

    for pattern in patterns:
        matches = list(re.finditer(pattern, stem))

        if not matches:
            continue

        match = matches[-1]
        year = match.group("year")
        year = year[-2:]
        month = match.group("month")
        day = match.group("day")

        return {
            "year": year,
            "month": month,
            "day": day,
            "date_token": f"{year}_{month}_{day}",
            "matched_text": match.group(0),
            "span": match.span(),
        }

    return None


def extract_sector_from_filename(file_name, date_match=None):
    stem = Path(file_name).stem
    working = stem

    if date_match:
        matched_text = date_match["matched_text"]
        working = working.replace(matched_text, " ")

    cleanup_patterns = [
        r"^GEOSP[-_ ]?TRN[-_ ]?\d+",
        r"\bGS\b",
        r"\bORTOFOTO\b",
        r"\bORTHOMOSAIC\b",
        r"\bORTOMOSAICO\b",
        r"\bDRONE\b",
        r"\bPAO\b",
        r"\(.*?\)",
    ]

    working = strip_accents(working).upper()

    for pattern in cleanup_patterns:
        working = re.sub(pattern, " ", working, flags=re.IGNORECASE)

    working = re.sub(r"[-_]+", " ", working)
    working = re.sub(r"\s+", " ", working).strip()

    return format_sector_token(working)


def build_expected_image_name(file_name, output_extension=DEFAULT_RENAMED_EXTENSION):
    date_match = extract_date_token_from_filename(file_name)

    if not date_match:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": None,
            "expected_sector": None,
            "rename_status": "sin_fecha",
        }

    sector = extract_sector_from_filename(file_name, date_match)

    if not sector:
        return {
            "expected_name": None,
            "expected_file_name": None,
            "expected_stem": None,
            "expected_date_token": date_match["date_token"],
            "expected_sector": None,
            "rename_status": "sin_sector",
        }

    expected_stem = f"{RENAME_PREFIX}_{date_match['date_token']}_{sector}"
    expected_file_name = f"{expected_stem}{output_extension}"

    return {
        "expected_name": expected_stem,
        "expected_file_name": expected_file_name,
        "expected_stem": expected_stem,
        "expected_date_token": date_match["date_token"],
        "expected_sector": sector,
        "rename_status": "ok",
    }


if input_images_df.empty:
    input_expected_names_df = input_images_df.copy()
    for column in ["expected_name", "expected_file_name", "expected_stem", "expected_date_token", "expected_sector", "rename_status"]:
        input_expected_names_df[column] = pd.Series(dtype="object")
else:
    expected_names_df = pd.DataFrame(
        [build_expected_image_name(file_name) for file_name in input_images_df["file_name"]]
    )
    input_expected_names_df = pd.concat([input_images_df.reset_index(drop=True), expected_names_df], axis=1)

print(input_expected_names_df["rename_status"].value_counts(dropna=False))
input_expected_names_df[["file_name", "relative_path", "expected_file_name", "expected_name", "expected_date_token", "expected_sector", "rename_status"]] if not input_expected_names_df.empty else input_expected_names_df

## Fase 1.6 - Comparar imagenes nuevas usando el nombre renombrado esperado

In [ ]:
def add_python_rename_match(input_expected_names_df, mosaic_inventory_df):
    if input_expected_names_df.empty:
        result_df = input_expected_names_df.copy()
        result_df["expected_name_exists_in_mosaic"] = pd.Series(dtype="bool")
        result_df["load_status"] = pd.Series(dtype="object")
        return result_df

    result_df = input_expected_names_df.copy()
    match_lookup = {}
    if not mosaic_inventory_df.empty:
        for _, mosaic_row in mosaic_inventory_df.iterrows():
            for key_column in ["mosaic_key", "mosaic_file_key", "mosaic_stem_key"]:
                key = normalize_image_key(mosaic_row.get(key_column))

                if key and key not in match_lookup:
                    match_lookup[key] = mosaic_row

    def find_mosaic_match(row):
        keys = [
            normalize_image_key(row.get("expected_file_name")),
            normalize_image_key(row.get("expected_stem")),
            normalize_image_key(row.get("expected_name")),
        ]

        for key in keys:
            if key and not pd.isna(key) and key in match_lookup:
                return match_lookup[key]

        return None

    result_df["mosaic_match"] = result_df.apply(find_mosaic_match, axis=1)
    result_df["expected_name_exists_in_mosaic"] = result_df["mosaic_match"].notna()
    result_df["matched_mosaic_source_field"] = result_df["mosaic_match"].map(lambda row: row.get("source_field") if row is not None else None)
    result_df["matched_mosaic_path"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_path") if row is not None else None)
    result_df["matched_mosaic_file_name"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_file_name") if row is not None else None)
    result_df["matched_mosaic_stem"] = result_df["mosaic_match"].map(lambda row: row.get("mosaic_stem") if row is not None else None)
    result_df = result_df.drop(columns=["mosaic_match"])
    result_df["load_status"] = result_df.apply(
        lambda row: "sin_nombre_esperado" if row["rename_status"] != "ok" else ("ya_cargada" if row["expected_name_exists_in_mosaic"] else "nueva_candidata"),
        axis=1,
    )

    return result_df


input_vs_mosaic_df = add_python_rename_match(input_expected_names_df, mosaic_image_inventory_df)

print(input_vs_mosaic_df["load_status"].value_counts(dropna=False) if not input_vs_mosaic_df.empty else "Sin imagenes en input")

review_columns = [
    "file_name",
    "relative_path",
    "expected_file_name",
    "expected_name",
    "expected_date_token",
    "expected_sector",
    "rename_status",
    "expected_name_exists_in_mosaic",
    "matched_mosaic_source_field",
    "matched_mosaic_file_name",
    "matched_mosaic_path",
    "load_status",
]

input_vs_mosaic_df[review_columns] if not input_vs_mosaic_df.empty else input_vs_mosaic_df

## Fase 1.7 - Exportar resultados para analisis

In [ ]:
def export_dataframe_csv(dataframe, output_folder, file_name):
    output_path = Path(output_folder) / file_name
    dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    return output_path


def export_dataframes_sqlite(dataframes, sqlite_path):
    import sqlite3

    sqlite_path = Path(sqlite_path)

    with sqlite3.connect(sqlite_path) as connection:
        for table_name, dataframe in dataframes.items():
            dataframe.copy().to_sql(table_name, connection, if_exists="replace", index=False)

    return sqlite_path


run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_results_dir = Path.cwd() / "outputs" / "carga_imagenes" / run_timestamp
output_results_dir.mkdir(parents=True, exist_ok=True)

result_tables = {
    "input_images": input_images_df,
    "mosaic_fields": mosaic_fields_df,
    "mosaic_sample": mosaic_df,
    "mosaic_image_inventory": mosaic_image_inventory_df,
    "input_expected_names": input_expected_names_df,
    "input_vs_mosaic": input_vs_mosaic_df,
}

if "new_images_df" in globals():
    result_tables["initial_new_images"] = new_images_df

exported_results = {
    "input_images_csv": export_dataframe_csv(input_images_df, output_results_dir, "01_input_images.csv"),
    "mosaic_fields_csv": export_dataframe_csv(mosaic_fields_df, output_results_dir, "02_mosaic_fields.csv"),
    "mosaic_sample_csv": export_dataframe_csv(mosaic_df, output_results_dir, "03_mosaic_sample.csv"),
    "mosaic_image_inventory_csv": export_dataframe_csv(mosaic_image_inventory_df, output_results_dir, "04_mosaic_image_inventory.csv"),
    "input_expected_names_csv": export_dataframe_csv(input_expected_names_df, output_results_dir, "05_input_expected_names.csv"),
    "input_vs_mosaic_csv": export_dataframe_csv(input_vs_mosaic_df, output_results_dir, "06_input_vs_mosaic.csv"),
}

if "new_images_df" in globals():
    exported_results["initial_new_images_csv"] = export_dataframe_csv(new_images_df, output_results_dir, "07_initial_new_images.csv")

summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "input_folder", "value": str(PATH_INPUT_SCAN)},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "input_images_count", "value": len(input_images_df)},
    {"metric": "mosaic_inventory_count", "value": len(mosaic_image_inventory_df)},
    {"metric": "expected_names_count", "value": len(input_expected_names_df)},
    {"metric": "comparison_count", "value": len(input_vs_mosaic_df)},
]

if not input_vs_mosaic_df.empty:
    for status, count in input_vs_mosaic_df["load_status"].value_counts(dropna=False).items():
        summary_rows.append({"metric": f"load_status_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)
result_tables["summary"] = summary_df
exported_results["summary_csv"] = export_dataframe_csv(summary_df, output_results_dir, "00_summary.csv")
exported_results["sqlite"] = export_dataframes_sqlite(
    result_tables,
    output_results_dir / f"carga_imagenes_{run_timestamp}.sqlite",
)

exported_results_df = pd.DataFrame(
    [{"name": name, "path": str(path)} for name, path in exported_results.items()]
)

print(f"Resultados exportados en: {output_results_dir}")
exported_results_df